<style>
:root{--navy:#253858;--blue:#1F6FEB;--gray:#667085;--light:#F5F7FA;--border:#D0D5DD}
.jp-RenderedHTMLCommon{font-family:"Noto Sans CJK SC","Microsoft YaHei",Arial,sans-serif;color:#101828;line-height:1.75}
.jp-RenderedHTMLCommon h1,.jp-RenderedHTMLCommon h2,.jp-RenderedHTMLCommon h3,.jp-RenderedHTMLCommon h4{color:var(--navy);font-family:"Noto Sans CJK SC","Microsoft YaHei",Arial,sans-serif}
.jp-RenderedHTMLCommon h2{margin-top:34px;border-bottom:2px solid #E4E7EC;padding-bottom:8px}
.jp-RenderedHTMLCommon p,.jp-RenderedHTMLCommon li{font-size:16px}
.jp-RenderedHTMLCommon table{font-size:14px;line-height:1.55}
.jp-RenderedHTMLCommon th{background:#EEF4FF;color:#253858}
.jp-RenderedHTMLCommon td,.jp-RenderedHTMLCommon th{vertical-align:top}
.jp-RenderedHTMLCommon code{font-size:90%}
.jp-RenderedHTMLCommon img{max-width:100%;height:auto;border:1px solid #D0D5DD;border-radius:10px;background:white;display:block;margin:10px auto 4px}
.callout{border-left:5px solid #1F6FEB;background:#EEF4FF;border-radius:8px;padding:14px 18px;margin:14px 0;color:#101828}
.callout-green{border-left:5px solid #2E7D32;background:#EDF7EE;border-radius:8px;padding:14px 18px;margin:14px 0;color:#101828}
.callout-warn{border-left:5px solid #B45309;background:#FFF4E5;border-radius:8px;padding:14px 18px;margin:14px 0;color:#101828}
.source{font-size:12px;color:#667085;line-height:1.5;margin-top:8px}
.caption{text-align:center;color:#667085;font-size:12px;margin:6px 0 16px}
.evidence-card{border:1px solid #D0D5DD;border-radius:10px;padding:14px 18px;margin:14px 0;background:#FFFFFF;box-shadow:0 2px 8px rgba(16,24,40,.05);color:#101828}
.evidence-card h4{margin:0 0 8px;color:#253858}
.evidence-meta{font-size:13px;color:#475467;background:#F2F4F7;border-radius:6px;padding:8px 10px;margin:8px 0}
.evidence-conclusion{border-left:4px solid #2E7D32;background:#EDF7EE;border-radius:6px;padding:10px 12px;margin-top:10px}
.evidence-limit{border-left:4px solid #B45309;background:#FFF4E5;border-radius:6px;padding:10px 12px;margin-top:10px}
.source-badge{display:inline-block;font-size:12px;font-weight:700;color:#344054;background:#EAECF0;border-radius:999px;padding:2px 9px;margin-right:6px}
.small-note{font-size:13px;color:#667085;line-height:1.6}
</style>

<div style="background:#253858;border-left:8px solid #1F6FEB;border-radius:16px;padding:34px 42px;color:white;margin:6px 0 20px">
  <div style="font-size:34px;font-weight:800;line-height:1.3">《搭建类似Hermes Agent的自进化长期记忆》</div>
  <div style="font-size:18px;line-height:1.7;margin-top:10px;color:#D8E0EE">从一次模型请求的记忆槽位出发，搭一套能写入、整理、升格、召回的长期记忆闭环。</div>
</div>


### 学习目标

概念导论的责任地图里有一块"会话、状态与记忆"。会话内的状态在窗口里就能维持，更麻烦的是另一半：**任务结束之后，哪些经历值得留下？留下之后怎样被修正、被遗忘？下一次任务开始时，又怎样把该带的带回来？** 本文从这个问题出发，搭一套类似 Hermes Agent 的长期记忆自进化闭环。

读完本文，应当能够：

- 用 `demo.py` 的槽位视角，说清一次模型请求里"记忆"到底是什么；
- 用四层记忆（raw / consolidated / semantic / working）解释经历怎样变成可复用记忆；
- 跑通两步压缩：会话压缩、升格长期记忆（含能力/方法记忆）；
- 分清两种触发：每轮对话完成后的检测、定时触发（crontab / 任务计划程序）；
- 分清两条召回：新会话召回、过程记忆召回，并看到带/不带记忆的输出差异。

### 阅读方式

概念块可以按需回看；代码块从上往下依次运行。正文里的代码只保留模型请求核心，输入输出走运行时变量；完整的文件读写版在 `scripts/`，和这些请求一一对应。

有一条分工原则贯穿全文：**理解自然语言的判断交给模型；可核对的结构化判断留在代码。**

## 一、同一个坑摔两次：缺少记忆的 Agent 长什么样

小李做了一个旅行规划 Agent。6 月中旬，用户让它排北京三日亲子游：初版把故宫和环球影城排进了同一天，路线检查工具报了失败；修正成"同区域聚合、远距离大项单独占一整天"之后检查通过。用户临走前还特意交代了一句："以后给我这种结果，把检查结论一起附上。"

一周后，同一个用户来排西安两日游。Agent 把兵马俑和回民街塞进了同一天——兵马俑在临潼区，距市区约 40 公里，这和上周北京那次是同一类错误。用户的"附检查结论"交代，它也像没听过一样。

这不是模型能力问题。会话结束时上下文被丢弃，上一次的失败证据、修正方案、用户交代，全都留在了那段再也不会被读到的聊天记录里。要解决它，第一反应往往是"上 RAG"——但先看一下 RAG 和记忆系统各自覆盖什么。


### 记忆系统有四个动作，RAG 只覆盖其中一个

RAG 回答的问题是"从一批资料里找相关片段"。资料本身是静态的：不管任务成败，它都不会变。记忆系统面对的是不断产生的经历，至少要回答四个问题：

| 动作 | 要回答的问题 | 常见工程实现 |
|---|---|---|
| ingestion 写入 | 什么经历进入记忆系统 | 事件流、工具结果、用户偏好、决策记录 |
| revision 修正 | 旧记忆怎样被改写 | 摘要重写、同义合并、重新关联证据 |
| forgetting 遗忘 | 什么不再进入候选范围 | 过期、降权、归档、删除 |
| retrieval 召回 | 新任务取回哪些记忆 | 文本检索、规则过滤、上下文打包 |

![RAG 与记忆系统](assets/01_rag_vs_memory.svg)

<div class="caption">图 1：RAG 只对应"召回"一格；另外三个动作决定"这条记忆值不值得信"。</div>

<div class="callout">
说白了：数据库负责"存进去还能读出来"；记忆系统还要负责"这条记录以后该不该影响输出"。后者要靠一套管理规则，不是换个存储库就能解决。
</div>


## 二、记忆的最小原理：一次模型请求是怎么"有记忆"的

`demo.py` 已经把答案演示得很直接：**模型本身一句话都不记得。它看起来有记忆，只是因为我们把要记的内容拼成文字、塞进了这一次请求里。**

`demo.py` 里的每个变量都是一个"记忆槽位"：

| demo.py 里的变量 | 是什么 | 谁来生产 |
|---|---|---|
| `chat_history` | 本轮会话的聊天记录 | 会话框架自动累积 |
| `memory` | 跨会话的长期记忆 | 先手写一条示例，随后改成系统自动生产 |
| `extra_info` | 当前任务的补充信息 | 任务侧临时提供 |
| `instruction` | 本次处理要注意什么 | 任务侧临时提供 |

先配置模型，然后把 `demo.py` 的核心重跑一遍。

In [ ]:
# ===== 准备模型 =====
import json
import os

from agently import Agently
from dotenv import find_dotenv, load_dotenv

# 从项目根目录的 .env 里读出模型密钥
load_dotenv(find_dotenv(usecwd=True))

api_key = os.getenv("DEEPSEEK_API_KEY")
if not api_key:
    raise RuntimeError("需要 DEEPSEEK_API_KEY（放在项目根目录 .env 或 shell 导出）。")

# 模型端点只配置一次，后面每个 create_agent() 都复用这份设置
Agently.set_settings(
    "OpenAICompatible",
    {
        "base_url": os.getenv("DEEPSEEK_BASE_URL", "https://api.deepseek.com/v1"),
        "api_key": api_key,
        "model": os.getenv("DEEPSEEK_DEFAULT_MODEL", "deepseek-chat"),
    },
)

# 演示里不读写文件：每一步的产物都存进这个字典，下一步再取出来用。
# 文件读写实现在 scripts/ 里，和这里的模型请求一一对应。
runtime_memory_store = {}
print("model configured")

In [ ]:
# ===== demo.py 核心演示：模型的"记忆"就是拼进请求的文字 =====

# 槽位 1：本轮会话的聊天记录（会话内上下文）
chat_history = [
    {"role": "user", "content": "帮我记一下，我今天要去超市买三个鸡蛋"},
    {"role": "assistant", "content": "好的，已经记下了！🥚"},
]

# 槽位 2：跨会话的长期记忆（现在先手写，随后由记忆系统自动生产）
memory = "用户偏好：回复要简短，喜欢用 emoji。"

# 槽位 3：当前任务的补充信息
extra_info = "Todo list: [ ] 今天要去超市买三个鸡蛋"

agent = Agently.create_agent()
result = await (
    agent
    .set_chat_history(chat_history)                        # 聊天记录放进请求
    .info({"重要记忆": memory, "信息补充": extra_info})     # 记忆槽位放进请求
    .input("刚才我们说了什么？我接下来还有什么要做的？")
    .async_start()
)
print(result)

把上面的 `memory` 变量删掉再跑一遍，模型就"忘了"用户偏好。记忆的全部秘密就是那段字。

剩下的问题只有一个：**这段字由谁、在什么时候、从哪里生产出来？** 这就要靠分层。

## 三、四层记忆理论：先分清放在哪里

CoALA（Cognitive Architectures for Language Agents）把语言 Agent 的记忆分成 working、episodic、semantic、procedural 四类：工作记忆、情景记忆、语义记忆、程序记忆。落到工程上，可以先拆成四个更容易操作的层级：

![四层记忆](assets/02_memory_tiers.svg)

<div class="caption">图 2：左边看记忆怎样生成——原始事件整理后升格为稳定规则；右边看记忆怎样使用——任务按需取回。</div>

| 层 | 放什么 | 默认是否进入上下文 | 文件产物 |
|---|---|---|---|
| Raw Episodic | 原始事件、工具输出、对话 | 否 | `materials/simulated_long_conversation.jsonl` |
| Consolidated Episodic | 会话摘要、候选记忆、过程记忆 | 候选 | `session_memory.json` |
| Semantic / Policy | 稳定规则、偏好、长期事实、能力方法 | 按任务召回 | `long_term_memory.json` / `capability_memory.json` |
| Working Memory | 当前任务必要信息 | 是 | `new_session_context.json` / `process_context.json` |

分层是为了三件事：只把少量稳定信息放进上下文；需要证据时能回读原始记录；记忆变旧或冲突时能定位并修正。raw 层只追加、不修饰；semantic 层少而稳定，每条都要有升格理由；working memory 是临时产物，用完可以重建。

Hermes Agent 的实际结构可以对进这张表：`MEMORY.md` / `USER.md` 对应稳定规则与用户偏好（semantic 层），SQLite + `session_search` 对应完整会话历史（raw 层），会话结束抽取和后台 review 对应整理动作，Skills 自改进对应能力/方法记忆。这套实现不照搬它的文件格式，只沿用分工：稳定层小而可信，历史层完整可查，后台整理负责把经历压成可复用经验。

## 四、生产：把经历压成记忆

`materials/simulated_long_conversation.jsonl` 里有三组会话（36 条事件）：

| 会话 | 内容 | 埋了什么 |
|---|---|---|
| `s_0614_beijing` | 北京三日亲子游 | 路线检查失败到修正、午睡偏好、显式交代以后附检查结论 |
| `s_0621_xian` | 西安两日游 | 同一类跨区失败第二次出现、重要景点排上午、口误噪声 |
| `s_0622_expense` | 出租车发票报销 | 另一个项目的长期要求，用来验证项目隔离 |

`scripts/01` 会读完整文件逐个会话压缩。下面先取北京会话里最关键的 4 条事件，看清楚机制。

In [ ]:
# ===== 原始会话事件(raw 层的一小段样例)=====
runtime_raw_events = [
    {
        "event_id": "evt_a05",
        "project_id": "travel-agent",
        "session_id": "s_0614_beijing",
        "role": "user",
        "text": "孩子才5岁,走不了太多路,下午两点到四点最好能回酒店午睡。",
    },
    {
        "event_id": "evt_a07",
        "project_id": "travel-agent",
        "session_id": "s_0614_beijing",
        "role": "tool",
        "tool": "route_checker",
        "status": "failed",
        "text": "路线检查失败:下午从故宫到环球影城通勤过长,对5岁儿童不可执行。",
    },
    {
        "event_id": "evt_a08",
        "project_id": "travel-agent",
        "session_id": "s_0614_beijing",
        "role": "assistant",
        "text": "调整思路:同区域景点聚合到同一天,远距离大项单独占一整天。",
    },
    {
        "event_id": "evt_a12",
        "project_id": "travel-agent",
        "session_id": "s_0614_beijing",
        "role": "user",
        "text": "以后给我这种结果的时候,把检查结论一起附上。",
    },
]

# 打印成对话的样子,方便看
for event in runtime_raw_events:
    if event.get("tool"):
        print(f"{event['role']} <{event['tool']}:{event['status']}>: {event['text']}")
    else:
        print(f"{event['role']}: {event['text']}")

### raw 流水不能直接当记忆

这 4 条事件覆盖了四类信息：用户偏好、工具失败、修正做法、显式长期要求。但直接把 raw 流水塞进下次请求不行："远距离大项单独占一整天"这条最有价值的经验，分散在失败报错、修正方案、用户确认几条事件里，没有任何一条能直接变成以后要遵守的规则。检索算法再好也没用——**raw 层里根本不存在这条整理后的记录**。

所以要压缩，分两步：

```text
raw 事件
  ↓ 第一步压缩（每个会话一次）
会话摘要 + 候选记忆 + 过程记忆     ← session_memory
  ↓ 第二步压缩（升格，宁紧勿松）
长期记忆（含能力/方法）             ← long_term_memory / capability_memory
```

第一步压缩有两个字段约束值得停一下：`statement` 要求"脱离本次对话也能读懂"——记忆是给未来的任务读的，"用户说下午要午睡"到了下个月就没人知道指的是谁；`evidence_event_ids` 强制每条候选挂上支撑它的事件——没有证据的记忆，后面既没法核对，也不该升格。

In [ ]:
# ===== 第一步压缩：raw 事件 -> session memory =====
# 约束尽量写进 .output() 的字段说明里：模型既能看懂，代码也能拿到稳定结构。
agent = Agently.create_agent()
session_memory_request = (
    agent.create_execution()
    .info({"对话事件流": runtime_raw_events})
    .input("对这段对话事件流做记忆抽取。")
    .output(
        {
            "summary": ("str", "三句话以内，概括这次会话做了什么、失败过什么、怎么修正的。"),
            "candidate_memories": [
                {
                    "kind": (
                        "str",
                        "枚举：user_preference | fact | lesson | skill。只抽取以后任务还会用得上的信息；寒暄、闲聊、口误不要抽。",
                    ),
                    "statement": ("str", "脱离本次对话也能读懂的一句话。"),
                    "durable": ("bool", "用户是否明确表达这条记忆以后一直有效。"),
                    "evidence_event_ids": [("str", "只能引用输入里出现过的 event_id。")],
                }
            ],
            "process_memory": [
                {
                    "note": ("str", "当前过程状态：失败的工具、待确认的点、已修正的做法。"),
                    "status": ("str", "枚举：open | resolved。"),
                    "evidence_event_ids": [("str", "只能引用输入里出现过的 event_id。")],
                }
            ],
        }
    )
)

print("===== Agently 实际发送给模型的 prompt =====")
print(session_memory_request.get_prompt_text())

session_memory = await session_memory_request.async_start()
runtime_memory_store["session_memory"] = session_memory
print(json.dumps(session_memory, ensure_ascii=False, indent=2))

### 大量历史怎么处理：分批抽取，再合并升格

上面只压了一个会话的 4 条事件。历史一多，不能把所有 raw 事件一次塞给模型——上下文装不下，失败也没法重跑。更稳的做法是拆成 Map-Reduce：

| 阶段 | 模型看到什么 | 输出什么 | 为什么这样切 |
|---|---|---|---|
| Map | 一个会话或一个时间片 | 会话摘要、候选记忆、过程记忆 | 上下文短，证据完整，失败可重跑 |
| Reduce | 多批候选记忆 | 合并后的长期记忆 | 同义合并、跨会话重复、证据核对 |
| Checkpoint | 已处理批次和 session | 断点文件 | 中断后从未完成批次继续 |

`scripts/01` 就是 Map 的最小实现（逐个会话循环压缩），`scripts/02` 是 Reduce。正文不重复写这个循环。

### 升格：三条通道，宁紧勿松

候选记忆还不是长期记忆。长期记忆会在未来任务里优先进入上下文，进去一条错的，以后每个任务都要为它买单。升格通道只有三条，满足任意一条才升：

| 升格通道 | 适合什么内容 | 例子 |
|---|---|---|
| 用户显式长期要求 | 用户明确说"以后都这样" | "以后输出行程要附检查结论" |
| 跨会话重复 | 多次独立出现的偏好或规则 | 多次亲子游都强调不要太累 |
| 工具失败证据 | 失败后形成的稳定修正规则 | 远距离大项不要塞进半天 |

升格时，候选的四类 `kind` 各归各位：

```text
user_preference → user_preference（用户偏好）
fact            → stable_fact（稳定事实）
lesson          → project_rule（项目规则）
skill           → capability_method（能力/方法，下一小节单独展开）
```

还有一个字段要注意：`recall_keywords` 由模型根据语义生成，是这条记忆自带的检索词。**召回不会用代码写死的词表**——固定关键词列表换个场景就失效，检索词应该从记忆本身和当前任务里长出来。

In [ ]:
# ===== 第二步压缩：候选记忆 -> 长期记忆 =====
# 升格门槛宁紧勿松：三条通道一条都不满足的候选，留在会话层等新证据。
agent = Agently.create_agent()
long_term_memory = await (
    agent
    .info({"session_memory": runtime_memory_store["session_memory"]})
    .input("判断哪些候选记忆可以升格为长期记忆。")
    .output(
        {
            "long_term_records": [
                {
                    "memory_id": ("str", "稳定、可追踪的记忆 id。"),
                    "project_id": ("str", "这条记忆所属项目。"),
                    "category": (
                        "str",
                        "枚举：user_preference | project_rule | stable_fact | capability_method。按候选 kind 归类：preference 仍是偏好，fact 归稳定事实，lesson 归项目规则，skill 归能力方法。",
                    ),
                    "statement": ("str", "只写跨会话仍然可能影响行为的内容。"),
                    "promotion_reason": (
                        "str",
                        "枚举：explicit_user_instruction | repeated_across_sessions | tool_failure_evidence。升格理由只能落在这三条通道里。",
                    ),
                    "recall_keywords": [("str", "根据语义生成的检索词；召回不用写死的词表。")],
                    "evidence": [("str", "支撑这条长期记忆的 event_id。")],
                }
            ]
        }
    )
    .async_start()
)

runtime_memory_store["long_term_memory"] = long_term_memory
print(json.dumps(long_term_memory, ensure_ascii=False, indent=2))

### 能力/方法记忆：把"会做的事"沉淀成方法卡

长期记忆不只有"用户喜欢什么"和"项目规则是什么"。系统长期运行还会积累一种更值钱的内容：**能力/方法记忆**——它可以视作长期记忆的一类（上一步的 `capability_method`），接近 CoALA 里的 procedural memory。它记录的是可复用做法，不是某一次任务的结论：

| 字段 | 含义 |
|---|---|
| `applies_when` | 什么场景下适用 |
| `method` | 具体步骤 |
| `validation` | 怎么确认做对了 |
| `failure_signals` | 什么现象说明这个方法失效 |
| `evidence` | 来自哪些事件 |

比如亲子旅行不是只记"用户要午休"，更值得留下的是一套方法：按地理区域聚合景点、远距离大项单独成天、重要景点放上午、生成草案后跑路线检查、输出里附检查结论。这套方法以后不只服务同一个用户，也能服务同一类任务。

上一步 skill 候选只有一句话，下面把它展开成完整的方法卡。

In [ ]:
# ===== 能力/方法记忆：从失败-修正-验证里抽方法 =====
agent = Agently.create_agent()
capability_memory = await (
    agent
    .info(
        {
            "session_memory": runtime_memory_store["session_memory"],
            "long_term_memory": runtime_memory_store["long_term_memory"],
        }
    )
    .input("从这些记忆里抽取值得长期保存的能力/方法。")
    .output(
        {
            "capability_methods": [
                {
                    "capability_id": ("str", "稳定、可追踪的方法 id。"),
                    "project_id": ("str", "方法所属项目。"),
                    "method_name": ("str", "方法名称。"),
                    "applies_when": ("str", "什么场景下适用；不要写成某一次任务的结论。"),
                    "method": [("str", "具体步骤，一步一条；优先来自失败后修正、工具验证、反复出现的成功做法。")],
                    "validation": [("str", "怎么确认做对了。")],
                    "failure_signals": [("str", "什么现象说明这个方法失效。")],
                    "recall_keywords": [("str", "用于召回这张方法卡的语义关键词。")],
                    "evidence": [("str", "支撑这张方法卡的 event_id；没有证据支撑的方法不要写。")],
                }
            ]
        }
    )
    .async_start()
)

runtime_memory_store["capability_memory"] = capability_memory
print(json.dumps(capability_memory, ensure_ascii=False, indent=2))

## 五、两种触发：什么时候激活压缩管线

第四节是"整理动作"本身；这一节只回答"什么时候激活它"。激活之后跑的就是第四节那两步压缩，不再展开。

![心跳整理管线](assets/03_heartbeat_pipeline.svg)

<div class="caption">图 3：心跳整理不是一次性魔法，而是扫描新记录、抽取记忆点、按证据分流、记录 checkpoint 的循环。</div>

**触发一：每轮对话完成后检测。** 一轮对话结束时，输入、模型输出、工具结果都齐了，做一次轻量检测，满足条件就投递整理任务。检测看的都是可计算信号，不需要模型：

| 信号 | 怎么检测 |
|---|---|
| 新事件数 | 当前 session 的事件数是否超过上次整理的位置 |
| 上下文压力 | 会话记录的 token 估算是否接近窗口阈值 |
| 过程变化 | 是否出现 failed tool、人工修正、用户明确说"以后都这样" |
| 写入锁 | 这个 session 是否已经有整理任务在跑（避免重复触发） |

**触发二：定时触发。** 低频会话、离线导入的历史不经过对话钩子，靠系统定时任务兜底。

macOS / Linux 用 crontab：

```bash
crontab -e    # 打开定时任务编辑器，加入下面这一行
```

```text
*/15 * * * * cd /path/to/Memory_v2 && python3 scripts/01_compress_session_memory.py && python3 scripts/02_promote_long_term_memory.py
```

意思是：每 15 分钟激活一次，先做会话压缩，再做升格。

Windows 用 schtasks（任务计划程序的命令行）：

```text
schtasks /Create /TN "memory_consolidation" /SC MINUTE /MO 15 /TR "cmd /c cd /d C:\path\to\Memory_v2 && python scripts\01_compress_session_memory.py && python scripts\02_promote_long_term_memory.py"
```

也可以打开图形界面的"任务计划程序"新建任务，设成每 15 分钟重复执行同一条命令。

两条触发线不是二选一：对话钩子抓即时信号（用户修正、工具失败、上下文将满），定时任务兜底低频会话和漏跑。

<div class="callout-warn">
安全提醒：后台自动写记忆要留三根钉子——来源标记（这条记忆从哪来）、证据链接（evidence 能追回原事件）、可回滚（写错了能定位撤销）。没有这三样，"自动整理"会变成"自动污染"（测量数据见资料来源里的 HEARTBEAT 论文）。
</div>

## 六、消费：召回分两条管线

"召回记忆"这个词太宽了。新任务刚开始时要召回的内容，和一个任务正在执行时要召回的内容，应该分开。更完整的工程实现里，Workspace 可以作为主存储和上下文打包层：

![基础召回](assets/04_hybrid_retrieval.svg)

<div class="caption">图 4：主存储保存事实来源和证据链；本次任务用 goal、scope、limit 决定哪些记忆进入上下文包。</div>

最小实现先用文件记忆库完成召回；换成 Workspace 或向量库时，变的是检索后端，不变的是"按当前任务打包上下文"这件事。

![召回流程](assets/05_recall_flow.svg)

<div class="caption">图 5：新会话召回跨会话长期记忆；过程召回只取当前会话里刚发生的失败、待确认和修正。</div>

**新会话召回**发生在新任务第一次请求模型之前，回答"这个新任务开始时，哪些长期记忆应该进入初始上下文"：

| 召回什么 | 例子 |
|---|---|
| 用户长期偏好 | 带 5 岁孩子旅行，下午要留午休窗口 |
| 项目规则 | 远距离大项不要和市区景点塞进同一天 |
| 能力/方法记忆 | 亲子行程先按区域聚合，再跑路线检查 |
| 少量相关会话摘要 | 上次类似任务怎么失败、怎么修正 |

它不召回整段原始聊天记录——新会话需要的是干净的上下文包，不是历史流水。

**过程记忆召回**发生在会话进行中（通常被一个工具失败触发），回答"当前执行卡住了，刚才发生过哪些状态、失败、待办需要继续带着"。它的范围是**当前这个会话自己的过程笔记**，不翻别的历史会话——跨会话的教训已经升格成长期记忆，归新会话召回管。

两条管线的第一步都一样：**模型先看当前情况生成检索关键词，代码再拿关键词去匹配**。区别在于输入的事件不同、匹配的记忆来源不同。先跑新会话召回：

In [ ]:
# ===== 新会话召回：新任务开始前，取回相关的长期记忆 =====
runtime_task = "带5岁的女儿去上海玩两天，孩子想去迪士尼乐园，帮我出一份可执行行程。"
records = runtime_memory_store["long_term_memory"]["long_term_records"]

# 第一步：让模型看"新任务 + 已有记忆的索引"，生成这次该用哪些检索词。
# 注意：关键词不是代码写死的，是模型现场生成的。
memory_index = []
for record in records:
    memory_index.append(
        {
            "memory_id": record["memory_id"],
            "category": record["category"],
            "statement": record["statement"],
            "recall_keywords": record["recall_keywords"],
        }
    )

agent = Agently.create_agent()
recall_query = await (
    agent
    .info({"新任务": runtime_task, "已有记忆索引": memory_index})
    .input("为这个新任务生成长期记忆的检索关键词。")
    .output(
        {
            "query_keywords": [("str", "贴近记忆索引里已有的表达，例如方法、限制、验证动作；不要只写新任务里的地点名。")],
            "preferred_categories": [("str", "这次优先需要召回的长期记忆类别。")],
            "reason": ("str", "一句话说明为什么这些词和类别适合当前任务。"),
        }
    )
    .async_start()
)
print("模型生成的检索词:", recall_query["query_keywords"])

# 第二步：代码拿着模型生成的关键词做匹配——代码只负责稳定执行，不做语义判断。
scored_records = []
for record in records:
    record_text = json.dumps(record, ensure_ascii=False)   # 整条记忆转成文字
    hits = 0
    for keyword in recall_query["query_keywords"]:
        if keyword and keyword in record_text:             # 命中一个关键词记一分
            hits += 1
    if hits > 0:                                           # 命中过的才召回
        scored_records.append({"命中数": hits, "record": record})

# 按命中数从多到少排，最多带 5 条进上下文
scored_records.sort(key=lambda item: item["命中数"], reverse=True)
selected_records = [item["record"] for item in scored_records[:5]]

runtime_memory_store["new_session_context"] = {
    "task": runtime_task,
    "recall_query": recall_query,
    "selected_long_term_records": selected_records,
}
print(json.dumps(runtime_memory_store["new_session_context"], ensure_ascii=False, indent=2))

### 过程记忆召回：只取当前执行需要的状态

现在换到会话进行中的场景：上海行程做到一半，路线检查工具又报了失败。这时候需要的不是重新加载所有长期规则，而是把**当前会话刚攒下的状态**找回来：刚失败的工具、待确认的点、刚修正过的做法。

下面的代码和新会话召回是同一套写法（模型生成关键词 → 代码匹配），对照着看两处不同：输入换成了当前事件，匹配对象换成了当前会话的过程笔记。

In [ ]:
# ===== 过程记忆召回：会话进行中被工具卡住时用 =====

# 当前会话（上海行程）进行到一半，刚发生的事件：
runtime_current_event = {
    "session_id": "s_0701_shanghai",
    "event": "route_checker_failed",
    "text": "路线检查又失败：把迪士尼和市区自然博物馆排在同一天，通勤时间过长。",
}

# 当前会话到此刻攒下的过程笔记（真实系统里它随对话逐轮累积）
runtime_process_memory = [
    {"note": "迪士尼这类远距离大项已确认单独占一整天，不和市区景点混排。", "status": "resolved"},
    {"note": "route_checker 判过'迪士尼+市区自然博物馆同一天'通勤过长，正在改。", "status": "open"},
    {"note": "还没和用户确认住在哪个区，第二天通勤估算暂按市中心。", "status": "open"},
    {"note": "本次输出要按用户长期要求附上路线检查结论。", "status": "open"},
]

# 第一步：还是模型先生成关键词——但这次它关注"刚发生的失败、待确认、修正"
agent = Agently.create_agent()
process_query = await (
    agent
    .info({"当前事件": runtime_current_event})
    .input("为过程记忆召回生成检索关键词。")
    .output(
        {
            "query_keywords": [("str", "围绕当前执行刚发生的失败、待确认、修正动作生成；不要生成长期偏好类的检索计划。")],
            "reason": ("str", "一句话说明这些词为什么适合当前过程召回。"),
        }
    )
    .async_start()
)
print("模型生成的检索词:", process_query["query_keywords"])

# 第二步：和新会话召回同一套匹配写法，但匹配对象换成当前会话的过程笔记
scored_notes = []
for note in runtime_process_memory:
    note_text = json.dumps(note, ensure_ascii=False)
    hits = 0
    for keyword in process_query["query_keywords"]:
        if keyword and keyword in note_text:
            hits += 1
    if hits > 0:
        scored_notes.append({"命中数": hits, "note": note})

scored_notes.sort(key=lambda item: item["命中数"], reverse=True)
selected_process = [item["note"] for item in scored_notes]

runtime_memory_store["process_context"] = {
    "current_event": runtime_current_event,
    "selected_process_memory": selected_process,
}
print(json.dumps(runtime_memory_store["process_context"], ensure_ascii=False, indent=2))

## 七、效果对比：带记忆和不带记忆，输出差在哪

回到第二节的 demo：那时"重要记忆"槽位里是手写的字符串。现在这个槽位可以换成记忆系统自动生产、自动召回的产物——接法完全一样，还是把文字放进 `.info()`。同一个任务各请求一次，对比输出：

In [ ]:
# ===== 请求 1：不带记忆，模型只看到任务本身 =====
agent = Agently.create_agent()
draft_without_memory = await (
    agent
    .input(runtime_task)
    .output(
        {
            "行程骨架": [("str", "每天一句话，说明主要安排。")],
            "输出附带": [("str", "行程之外附带的提示。")],
        }
    )
    .async_start()
)

# ===== 请求 2：带记忆，把召回产物填进"记忆槽位"（对应 demo.py 里的 memory 变量）=====
agent = Agently.create_agent()
draft_with_memory = await (
    agent
    .info(
        {
            "必须遵守的长期记忆": runtime_memory_store["new_session_context"]["selected_long_term_records"],
            "可复用的方法": runtime_memory_store["capability_memory"]["capability_methods"],
        }
    )
    .input(runtime_task)
    .output(
        {
            "行程骨架": [("str", "每天一句话，说明主要安排；长期记忆要逐条落实到行程里。")],
            "输出附带": [("str", "行程之外附带的提示；要能看出长期记忆或方法卡的落点。")],
        }
    )
    .async_start()
)

print("===== 不带记忆 =====")
print(json.dumps(draft_without_memory, ensure_ascii=False, indent=2))
print("===== 带记忆 =====")
print(json.dumps(draft_with_memory, ensure_ascii=False, indent=2))

带记忆的输出应该能看出三处差异：午休窗口被排进了日程、迪士尼单独占了一整天、输出附带里有检查结论的位置。验收标准很简单：**同一个任务，带记忆和不带记忆，输出有看得见的差异。**

## 八、工程边界：谁负责什么，谁不负责什么

| 角色 | 负责什么 | 不负责什么 |
|---|---|---|
| Session | 当前会话历史、窗口裁剪 | 跨任务长期记忆的整理和取用 |
| 文件记忆库 | session_memory、long_term_memory 等中间产物 | 工程级权限、索引、并发和查询优化 |
| 触发器（对话钩子 / 定时任务） | 判断什么时候激活整理 | 整理动作本身、记忆理论和业务规则 |
| 模型 | 抽取、同义判断、摘要、生成检索关键词 | 独自决定哪些记忆可信 |
| 代码 | 稳定执行匹配、核对证据等结构化事实 | 用关键词规则替模型做语义判断 |

工程再往下走，还有两个可选扩展：**Workspace** 负责持久化记录、scope 隔离和上下文包生成；**向量库**负责语义近义召回加速。它们都不改变记忆文件格式和分工边界。

## 总结

1. **记忆的最小原理**：模型不记事，"记忆"就是拼进这一次请求的文字（demo.py 的槽位）；整套记忆系统做的事，就是自动生产和挑选这段文字；
2. **生产靠两步压缩**：会话压缩（摘要+候选+过程记忆）、升格长期记忆（三条通道，宁紧勿松）；能力/方法是长期记忆的一类，把"会做的事"沉淀成方法卡；
3. **输出契约要长在 `.output()` 里**：字段含义、枚举范围、证据要求写进字段注释，Agently 会把它们渲染进实际 prompt；需要观察时可以先 `get_prompt_text()` 再执行；
4. **触发分两种**：每轮对话完成后的轻量检测抓即时信号，crontab / 任务计划程序兜底低频和漏跑；激活的都是同一套压缩管线；
5. **召回分两条**：新会话召回取跨会话的长期记忆，过程召回取当前会话的临时状态；两条管线都是模型生成关键词、代码稳定执行匹配；
6. **验收看行为**：同一个任务，带记忆和不带记忆，输出要有看得见的差异。

## 资料来源

- Hermes Agent Persistent Memory: https://hermes-agent.nousresearch.com/docs/user-guide/features/memory
- Hermes Agent Memory Providers: https://hermes-agent.nousresearch.com/docs/user-guide/features/memory-providers
- Cognitive Architectures for Language Agents (CoALA): https://arxiv.org/abs/2309.02427
- Generative Agents: Interactive Simulacra of Human Behavior: https://arxiv.org/abs/2304.03442
- Mind Your HEARTBEAT! Claw Background Execution Inherently Enables Silent Memory Pollution: https://arxiv.org/abs/2603.23064
- Chroma Docs: https://docs.trychroma.com/docs/overview/introduction
- LanceDB Hybrid Search: https://docs.lancedb.com/search/hybrid-search
